# Missing TermpointToENB 4G--4G
### 1 - Usar o EutranCell_TDD_FDD para criar uma lista virtual de co-sites
### 2 - Nessa lista virtual o site "MID(eNB,2,3) + UF" da src e da tgt são iguais, porém o eNB_name será diferente.
### 3 - Usar o MO TermpointToENB para obter os Termpoints 4G--4G existentes.
### 4 - Comparar a lista virtual com o Termpoint para identificarmos os missings

In [ ]:
from datetime import datetime
semana=datetime.today().strftime('%U')
semana_atual= int(semana) + 1
semana_atual

In [ ]:
import pandas as pd

df= pd.read_csv('C:/Camilo/EUtranCell_TDD_FDD.txt', sep= '\t', usecols=['UF', 'MUNICIPIO','eNB', 'CGI', 'Tipo'],encoding= 'ISO-8859-1')

df.drop_duplicates(inplace=True)
df['Site']= df['eNB'].str[1:4] + df['eNB'].str[-3:]

# Extraindo a parte desejada da coluna 'CGI'
df['CGI'] = df['CGI'].str.rsplit(pat='-', n=1).str[0]

df.head()

In [ ]:
eNB_src= df.copy()
eNB_tgt= df.copy()

lista_virtual= pd.merge(eNB_src, eNB_tgt, on=['UF', 'Site'], suffixes=['_src', '_tgt'], how='inner')
lista_virtual= lista_virtual[lista_virtual['eNB_src']!=lista_virtual['eNB_tgt']]

lista_virtual.drop_duplicates(subset=['UF','eNB_src','CGI_src', 'eNB_tgt', 'CGI_tgt'], inplace=True)

lista_virtual.head()

### Pegando as informações do MO TermpointToENodeB e considerando apenas operationalState= ENABLED para pegar tambem os casos de termpoints nao operacionais.

In [ ]:
arquivo_csv= 'C:/Camilo/' + 'TermPointToENB'+ '_'+str(semana_atual)+'.txt'
TermpointToENB= pd.read_csv(arquivo_csv, sep='\t' ,usecols=['UF', 'eNB', 'TermPointToENBId', 'Ancora','operationalState'], encoding='ISO-8859-1')
Termpoint= TermpointToENB.copy()
TermpointToENB= TermpointToENB[TermpointToENB['operationalState']=='ENABLED']
TermpointToENB.head()

### Comparando a lista virtual com o TermpointToENodeB para identificarmos os missings

In [ ]:
missing= pd.merge(lista_virtual, TermpointToENB, left_on=['UF','eNB_src', 'CGI_tgt'], right_on=['UF','eNB', 'TermPointToENBId'], how='left')

missing= missing[missing['Ancora'].isna()]

missing['Regional']= missing['UF'].replace(['AM', 'AP', 'MA', 'PA', 'RR', 'BA', 'SE'], ['N', 'N', 'N', 'N', 'N', 'BASE', 'BASE'])

missing.drop(columns=['MUNICIPIO_tgt'], inplace=True)

missing.rename(columns={'MUNICIPIO_src': 'MUNICIPIO'}, inplace=True)

# Usando pop para remover a coluna 'Regional'
regional = missing.pop('Regional')

# Inserindo 'Regional' na primeira posição
missing.insert(0, 'Regional', regional)

missing.head()

# Pegando as informações de IP address para usar na macro ".mos"

In [ ]:
import pandas as pd

# Carregando os dados
ipaddress = pd.read_csv('C:/Dumps/LTE_Total/AddressIPv4.txt', sep='\t', usecols=['eNB', 'Router', 'IP_address'])

# Criando um rank para priorizar LTE_X2 sobre LTE_OAM
ipaddress['Rank'] = ipaddress['Router'].apply(lambda x: 0 if x == 'LTE_X2' else (1 if x == 'LTE_OAM' else 2))

# Ordenando os dados pela coluna 'eNB' e 'Rank'
ipaddress.sort_values(by=['eNB', 'Rank'], inplace=True)

# Removendo duplicatas mantendo a entrada com menor 'Rank' para cada 'eNB'
ipaddress = ipaddress.drop_duplicates(subset=['eNB'], keep='first')

# Removendo a coluna 'Rank' pois não é mais necessária
ipaddress.drop(columns=['Rank'], inplace=True)

# Agora o DataFrame ipaddress terá apenas uma linha por eNB, priorizando LTE_X2 quando disponível, senão LTE_OAM
ipaddress.rename(columns={'eNB':'eNB_tgt'},inplace=True)
ipaddress.head()

In [ ]:
# Fazendo o join do missing TermpointToGNB com o IP_address

missing1= pd.merge(missing, ipaddress, on= 'eNB_tgt', how='left')
missing1.drop(columns='TermPointToENBId',inplace=True)
missing1['PLMN']= "mcc="+missing1['CGI_tgt'].str[:3] + ',mnc=' + missing1['CGI_tgt'].str[3:5] + ',mnclength=2'
missing1.head()

In [ ]:
Termpoint.head()

In [ ]:
# Adicionando o operationalState do Termpoint para sabermos se ele existe, mas esta disabled

missing2= pd.merge(missing1, Termpoint, left_on=['UF', 'eNB_src', 'eNB_tgt', 'CGI_tgt'], right_on=['UF', 'eNB','Ancora', 'TermPointToENBId'], suffixes=['_dir', '_tgt'], how='left')
missing2.drop(columns=['eNB_dir','Ancora_dir','operationalState_dir', 'eNB_tgt', 'Ancora_tgt'],inplace=True)
missing2.rename(columns={'operationalState_tgt': 'operationalState_ancora'},inplace=True)
missing2.head()

In [ ]:
missing2.to_excel('C:/Vivo Reports/LTE/LTE Mobilidade/Missing_TermPointToENB_4G_4G.xlsx', index=None)

print('Missing_TermPointToENB_4G_4G, Done')